In [1]:
#Import all packages
import os
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
import contextily as ctx
import scipy.stats
import scipy.interpolate
import tqdm
from pathlib import Path
import xarray as xr
import skgstat as skg
import seaborn as sns
import pysal
from pysal.explore import esda
from pysal.lib import weights
from splot.esda import moran_scatterplot
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from matplotlib_scalebar.scalebar import ScaleBar
import rasterio
from rasterio.plot import show
import rasterio
import xarray as xr
import numpy as np
from pathlib import Path
import re
from datetime import datetime
import geopandas as gpd
import re
from datetime import datetime
import numpy as np
import xarray as xr
import rasterio
from rasterio.features import rasterize
from shapely.geometry import mapping
from shapely.geometry import LineString, Point, MultiPoint, GeometryCollection
import shapely.ops as ops
import pandas as pd
import numpy as np
import geopandas as gpd

/opt/conda/lib/python3.11/site-packages/spaghetti/network.py:41: FutureWarning: The next major release of pysal/spaghetti (2.0.0) will drop support for all ``libpysal.cg`` geometries. This change is a first step in refactoring ``spaghetti`` that is expected to result in dramatically reduced runtimes for network instantiation and operations. Users currently requiring network and point pattern input as ``libpysal.cg`` geometries should prepare for this simply by converting to ``shapely`` geometries.
  warnings.warn(dep_msg, FutureWarning, stacklevel=1)


In [2]:
#Summon the Data
DATA = Path("/home/jovyan/Society_of_Bouy_Cowboys/Data")

NETCDF = DATA / "NET_CDF"
ICE = DATA / "Ice_edge"
SAT = DATA / "SAT_GeoTiff"

# buoy files
moor_files = {
    "S1P1": NETCDF / "S1P1.nc",
    "S1P2": NETCDF / "S1P2.nc",
    "S1P3": NETCDF / "S1P3.nc",
    "S1A1": NETCDF / "S1A1.nc"
}

In [3]:
#Create XArray for all GeoTiffs with Ice Edge

# Load all GeoTIFFs into a single xarray Dataset with a time dimension
scenes = {}
for tif_path in sorted(SAT.glob("*.tif")):
    m = re.search(r"_(\d{8})(\d{4})_", tif_path.stem)
    if m:
        t = datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M")
    else:
        continue
    with rasterio.open(tif_path) as src:
        scenes[t] = {
            "sigma0":    src.read(1),
            "png":       src.read(2),
            "bounds":    src.bounds,
            "transform": src.transform,
            "shape":     (src.height, src.width),
        }

# Parse datetime from filename e.g. ice_edge_201911260350.geojson
ice_edges = {}
for geojson_path in sorted(ICE.glob("*.geojson")):
    m = re.search(r"_(\d{8})(\d{4})", geojson_path.stem)
    if m:
        t = datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M")
        ice_edges[t] = gpd.read_file(geojson_path).to_crs("EPSG:32604")

print(f"Loaded {len(scenes)} SAR scenes")
print(f"Loaded {len(ice_edges)} ice edge files")

# ── Sort by time ───────────────────────────────────────────────────────────
times  = sorted(scenes.keys())
bounds = scenes[times[0]]["bounds"]
shape  = scenes[times[0]]["shape"]

# ── Rasterize ice edges onto the SAR grid ──────────────────────────────────
# For each SAR scene, if a matching ice edge exists (within 1 hour),
# burn it into a binary raster (1 = ice edge, 0 = no data).
# If no ice edge exists for that scene, fill with NaN.

def find_nearest_edge(t, ice_edges, max_hours=1):
    """Return the ice edge GeoDataFrame closest in time to t, within max_hours."""
    best_t, best_dt = None, None
    for et in ice_edges:
        dt = abs((t - et).total_seconds()) / 3600
        if dt <= max_hours and (best_dt is None or dt < best_dt):
            best_t, best_dt = et, dt
    return ice_edges[best_t] if best_t else None

ice_rasters = []
for t in times:
    edge_gdf = find_nearest_edge(t, ice_edges, max_hours=1)
    if edge_gdf is not None and len(edge_gdf) > 0:
        # Burn the line geometry into the raster grid
        burned = rasterize(
            [(mapping(geom), 1) for geom in edge_gdf.geometry],
            out_shape = shape,
            transform = scenes[t]["transform"],
            fill      = 0,
            dtype     = np.float32,
        )
    else:
        burned = np.full(shape, np.nan, dtype=np.float32)
    ice_rasters.append(burned)

print(f"Ice edge rasters built: {sum(np.any(r == 1) for r in ice_rasters)}/{len(times)} scenes have an ice edge")

# ── Build xarray Dataset ───────────────────────────────────────────────────
SAT_ds = xr.Dataset(
    data_vars=dict(
        SAR_backscatter=(["time", "y", "x"],
                np.stack([scenes[t]["sigma0"] for t in times]),
                {"long_name": "SAR backscatter", "units": "linear"}),
        ice_edge       =(["time", "y", "x"],
                np.stack(ice_rasters),
                {"long_name": "Ice edge (rasterized)", "units": "binary 0/1"}),
    ),
    coords=dict(
        time=(["time"], np.array(times, dtype="datetime64[ns]")),
        x   =(["x"],    np.linspace(bounds.left,   bounds.right, shape[1])),
        y   =(["y"],    np.linspace(bounds.top,     bounds.bottom, shape[0])),
    ),
    attrs=dict(crs="EPSG:32604"),
)

SAT_ds

Loaded 25 SAR scenes
Loaded 8 ice edge files
Ice edge rasters built: 9/25 scenes have an ice edge


<xarray.Dataset> Size: 801MB
Dimensions:          (time: 25, y: 2001, x: 2001)
Coordinates:
  * time             (time) datetime64[ns] 200B 2019-11-07T17:48:00 ... 2019-...
  * x                (x) float64 16kB 3.321e+05 3.321e+05 ... 4.374e+05
  * y                (y) float64 16kB 7.86e+06 7.86e+06 ... 7.755e+06 7.755e+06
Data variables:
    SAR_backscatter  (time, y, x) float32 400MB nan nan nan nan ... nan nan nan
    ice_edge         (time, y, x) float32 400MB nan nan nan nan ... nan nan nan
Attributes:
    crs:      EPSG:32604

In [4]:
def get_mooring_latlon(ncfile):
    ds = xr.open_dataset(ncfile)

    lat = np.asarray(ds["lat_lagrangian"]).astype(float).ravel()
    lon = np.asarray(ds["lon_lagrangian"]).astype(float).ravel()

    good = np.isfinite(lat) & np.isfinite(lon)

    lat = lat[good]
    lon = lon[good]

    if len(lat) == 0:
        raise ValueError(f"No valid lat/lon found in {ncfile}")

    # median gives robust representative location
    return np.median(lat), np.median(lon)

rows = []

for name, fn in moor_files.items():
    lat, lon = get_mooring_latlon(fn)
    rows.append({
        "source": name,
        "lat": lat,
        "lon": lon
    })

moor_df = pd.DataFrame(rows)

moor_gdf = gpd.GeoDataFrame(
    moor_df,
    geometry=gpd.points_from_xy(moor_df["lon"], moor_df["lat"]),
    crs="EPSG:4326"
)

# convert buoys into analysis CRS
moor_gdf = moor_gdf.to_crs("EPSG:32604")

moor_gdf

,source,lat,lon,geometry
0,S1P1,70.34613,-162.05704,POINT (385288.486 7807356.201)
1,S1P2,70.39473,-162.13675,POINT (382579.19 7812922.163)
2,S1P3,70.43995,-162.20300,POINT (380366.746 7818088.385)
3,S1A1,70.48695,-162.28278,POINT (377672.411 7823482.253)


In [5]:
#For Each Mooring Site, create XArray 
S1P1_ds= xr.open_dataset('/home/jovyan/Society_of_Bouy_Cowboys/Data/NET_CDF/S1P1.nc')

S1P2_ds= xr.open_dataset('/home/jovyan/Society_of_Bouy_Cowboys/Data/NET_CDF/S1P2.nc')

S1P3_ds= xr.open_dataset('/home/jovyan/Society_of_Bouy_Cowboys/Data/NET_CDF/S1P3.nc')

S1P4_ds= xr.open_dataset('/home/jovyan/Society_of_Bouy_Cowboys/Data/NET_CDF/S1P4.nc')

S1A1_ds= xr.open_dataset('/home/jovyan/Society_of_Bouy_Cowboys/Data/NET_CDF/S1A1.nc')

ds_list = [S1P1_ds, S1P2_ds, S1P3_ds, S1A1_ds]

In [6]:
xr.concat(ds_list, dim='source')

<xarray.Dataset> Size: 43MB
Dimensions:         (source: 4, time: 2604, freq: 84)
Coordinates:
  * time            (time) datetime64[ns] 21kB 2019-11-09T07:27:07.000002816 ...
  * freq            (freq) float64 672B 0.009766 0.009766 ... 0.4902 0.4902
Dimensions without coordinates: source
Data variables: (12/13)
    lat_lagrangian  (source, time) float64 83kB nan 70.35 nan ... nan 70.49
    lon_lagrangian  (source, time) float64 83kB nan -162.1 nan ... nan -162.3
    watertemp       (source, time) float64 83kB nan 20.76 nan ... nan nan nan
    sigwaveheight   (source, time) float64 83kB nan nan nan ... 1.709 nan 1.962
    peakwaveperiod  (source, time) float64 83kB nan nan nan ... 7.211 nan 7.211
    peakwavedirT    (source, time) float64 83kB nan 9.999e+03 nan ... nan nan
    ...              ...
    a1              (source, freq, time) float64 7MB nan nan nan ... nan nan nan
    b1              (source, freq, time) float64 7MB nan nan nan ... nan nan nan
    a2              (source, freq, time) float64 7MB nan nan nan ... nan nan nan
    b2              (source, freq, time) float64 7MB nan nan nan ... nan nan nan
    check           (source, freq, time) float64 7MB nan nan nan ... nan nan nan
    depth           (source, time) float64 83kB nan 13.0 nan nan ... nan nan nan

In [7]:
S1P3_ds.time

<xarray.DataArray 'time' (time: 749)> Size: 6kB
array(['2019-11-09T07:30:50.000000256', '2019-11-09T08:00:50.000003840',
       '2019-11-09T08:30:49.999996928', ..., '2019-11-24T20:30:49.999996928',
       '2019-11-24T21:00:50.000000256', '2019-11-24T21:30:50.000003840'],
      dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 6kB 2019-11-09T07:30:50.000000256 ... 2019...
Attributes:
    long_name:      time
    standard_name:  time

In [8]:
common_time = pd.date_range(
    S1A1_ds.time.min().values,
    S1A1_ds.time.max().values,
    freq="30min"
)

S1A1_ds_halfhour = S1A1_ds.interp(time=common_time)
S1A1_ds_halfhour

<xarray.Dataset> Size: 1MB
Dimensions:         (time: 725, freq: 42)
Coordinates:
  * freq            (freq) float64 336B 0.009766 0.02148 ... 0.4785 0.4902
  * time            (time) datetime64[ns] 6kB 2019-11-09T20:17:13.250987264 ....
Data variables:
    lat_lagrangian  (time) float64 6kB 70.49 70.49 70.49 ... 70.49 70.49 70.49
    lon_lagrangian  (time) float64 6kB -162.3 -162.3 -162.3 ... -162.3 -162.3
    sigwaveheight   (time) float64 6kB 0.656 0.6704 0.6849 ... 1.709 1.836 1.962
    peakwaveperiod  (time) float64 6kB 7.877 7.544 7.211 ... 7.211 7.211 7.211
    peakwavedirT    (time) float64 6kB nan nan nan nan nan ... nan nan nan nan
    energy          (freq, time) float64 244kB 0.0 0.0 0.0 ... 0.03249 0.04849
    a1              (freq, time) float64 244kB nan nan nan nan ... nan nan nan
    b1              (freq, time) float64 244kB nan nan nan nan ... nan nan nan
    a2              (freq, time) float64 244kB nan nan nan nan ... nan nan nan
    b2              (freq, time) float64 244kB nan nan nan nan ... nan nan nan
    check           (freq, time) float64 244kB nan nan nan nan ... nan nan nan

In [9]:
ds_list = [S1P1_ds, S1P2_ds, S1P3_ds, S1A1_ds_halfhour]
labels = ["S1P1", "S1P2", "S1P3", "S1A1"]

# common 30-minute grid over the overlapping time range
tmin = max(pd.Timestamp(ds.time.min().values).ceil("30min") for ds in ds_list)
tmax = min(pd.Timestamp(ds.time.max().values).floor("30min") for ds in ds_list)
common_time = pd.date_range(tmin, tmax, freq="30min")

# snap each dataset to the shared grid
aligned = [
    ds.reindex(time=common_time, method="nearest", tolerance=pd.Timedelta("15min"))
    for ds in ds_list
]

# combine
ds_out = xr.concat(
    aligned,
    dim=xr.DataArray(labels, dims="source", name="source")
)
ds_out

<xarray.Dataset> Size: 12MB
Dimensions:         (source: 4, time: 718, freq: 84)
Coordinates:
  * time            (time) datetime64[ns] 6kB 2019-11-09T20:30:00 ... 2019-11...
  * freq            (freq) float64 672B 0.009766 0.009766 ... 0.4902 0.4902
  * source          (source) <U4 64B 'S1P1' 'S1P2' 'S1P3' 'S1A1'
Data variables: (12/13)
    lat_lagrangian  (source, time) float64 23kB 70.35 70.35 ... 70.49 70.49
    lon_lagrangian  (source, time) float64 23kB -162.1 -162.1 ... -162.3 -162.3
    watertemp       (source, time) float64 23kB 15.82 15.73 14.57 ... nan nan
    sigwaveheight   (source, time) float64 23kB nan nan nan ... 1.89 1.827 1.823
    peakwaveperiod  (source, time) float64 23kB nan nan nan ... 7.211 7.544
    peakwavedirT    (source, time) float64 23kB nan nan nan nan ... nan nan nan
    ...              ...
    a1              (source, freq, time) float64 2MB nan nan nan ... nan nan nan
    b1              (source, freq, time) float64 2MB nan nan nan ... nan nan nan
    a2              (source, freq, time) float64 2MB nan nan nan ... nan nan nan
    b2              (source, freq, time) float64 2MB nan nan nan ... nan nan nan
    check           (source, freq, time) float64 2MB nan nan nan ... nan nan nan
    depth           (source, time) float64 23kB 13.0 13.0 13.0 ... nan nan nan

In [10]:
# ------------------------------------------------------------
# Load geometry definitions
# ------------------------------------------------------------

moorings_file = DATA / "QGIS:CoordinateSystem/moorings.geojson"
zero_file = DATA / "QGIS:CoordinateSystem/zerozero.geojson"

moor_gdf = gpd.read_file(moorings_file)
zero_gdf = gpd.read_file(zero_file)

# confirm CRS
moor_gdf = moor_gdf.to_crs("EPSG:32604")
zero_gdf = zero_gdf.to_crs("EPSG:32604")

zero_point = zero_gdf.geometry.iloc[0]

moor_gdf

,source,lat,lon,geometry
0,S1P1,70.34613,-162.05704,POINT (385288.486 7807356.201)
1,S1P2,70.39473,-162.13675,POINT (382579.19 7812922.163)
2,S1P3,70.43995,-162.20300,POINT (380366.746 7818088.385)
3,S1A1,70.48695,-162.28278,POINT (377672.411 7823482.253)


In [11]:
# ------------------------------------------------------------
# Define transect along mooring array
# ------------------------------------------------------------

# make sure these came from the geojson files
moor_order = ["S1P1", "S1P2", "S1P3", "S1A1"]

moor_line_gdf = moor_gdf.set_index("source").loc[moor_order].reset_index()
zero_point = zero_gdf.geometry.iloc[0]

# build transect explicitly starting at zero point
transect = LineString([zero_point] + list(moor_line_gdf.geometry))

# zero point is the start of the line
s0 = 0.0

# compute distances along this line
moor_line_gdf["moor_dist_m"] = moor_line_gdf.geometry.apply(transect.project)
moor_line_gdf["moor_dist_km"] = moor_line_gdf["moor_dist_m"] / 1000.0

print("zero_point:", zero_point)

moor_line_gdf[["source", "moor_dist_m", "moor_dist_km"]]

zero_point: POINT (387833.59575019206 7802001.769798128)


,source,moor_dist_m,moor_dist_km
0,S1P1,5928.534141,5.928534
1,S1P2,12118.866874,12.118867
2,S1P3,17738.898873,17.738899
3,S1A1,23768.264612,23.768265


In [12]:
# ------------------------------------------------------------
# 2) Helper functions to convert ice-edge geometry to
#    distance along the same 1D transect
# ------------------------------------------------------------
def _extract_points_from_intersection(intersection):
    """
    Return a list of shapely Points from an intersection geometry.
    Handles Point, MultiPoint, LineString, MultiLineString, GeometryCollection.
    """
    pts = []

    if intersection.is_empty:
        return pts

    gtype = intersection.geom_type

    if gtype == "Point":
        pts = [intersection]

    elif gtype == "MultiPoint":
        pts = list(intersection.geoms)

    elif gtype == "LineString":
        # if the edge overlaps the transect for a segment, use the midpoint
        pts = [intersection.interpolate(0.5, normalized=True)]

    elif gtype == "MultiLineString":
        pts = [g.interpolate(0.5, normalized=True) for g in intersection.geoms]

    elif gtype == "GeometryCollection":
        for g in intersection.geoms:
            pts.extend(_extract_points_from_intersection(g))

    return pts


def ice_edge_distance_along_transect(edge_gdf, transect, s0, target_s=None):
    """
    Returns the ice-edge distance (meters) along the transect
    relative to the 0-point. If multiple intersections exist,
    choose the one closest to target_s.
    """
    if edge_gdf is None or len(edge_gdf) == 0:
        return np.nan

    if edge_gdf.crs is None:
        raise ValueError("edge_gdf has no CRS")

    if str(edge_gdf.crs) != "EPSG:32604":
        edge_gdf = edge_gdf.to_crs("EPSG:32604")

    edge_union = edge_gdf.geometry.union_all() if hasattr(edge_gdf.geometry, "union_all") else edge_gdf.unary_union
    inter = transect.intersection(edge_union)

    pts = _extract_points_from_intersection(inter)

    if len(pts) == 0:
        return np.nan

    s_candidates = np.array([transect.project(pt) - s0 for pt in pts])

    if target_s is None:
        target_s = 0.0

    return s_candidates[np.argmin(np.abs(s_candidates - target_s))]

In [13]:
# ------------------------------------------------------------
# 3) Build a time series of ice-edge distance along transect
#    for each SAT scene time
# ------------------------------------------------------------
target_s = float(moor_line_gdf["moor_dist_m"].mean())  # choose crossing nearest array center

ice_dist_rows = []

for t in pd.to_datetime(SAT_ds.time.values):
    t_py = pd.Timestamp(t).to_pydatetime()

    edge_gdf = find_nearest_edge(t_py, ice_edges, max_hours=1)

    ice_dist_m = ice_edge_distance_along_transect(
        edge_gdf=edge_gdf,
        transect=transect,
        s0=s0,
        target_s=target_s
    )

    ice_dist_rows.append({
        "time": pd.Timestamp(t),
        "ice_edge_dist_m": ice_dist_m,
        "ice_edge_dist_km": ice_dist_m / 1000.0 if np.isfinite(ice_dist_m) else np.nan
    })

ice_dist_df = pd.DataFrame(ice_dist_rows)
ice_dist_df

,time,ice_edge_dist_m,ice_edge_dist_km
0,2019-11-07 17:48:00,NaN,NaN
1,2019-11-08 04:15:00,NaN,NaN
2,2019-11-08 04:19:00,NaN,NaN
3,2019-11-08 17:40:00,NaN,NaN
4,2019-11-09 03:46:00,NaN,NaN
5,2019-11-11 04:27:00,NaN,NaN
6,2019-11-14 17:40:00,NaN,NaN
7,2019-11-15 04:11:00,3802.467632,3.802468
8,2019-11-16 03:41:00,14439.562334,14.439562
9,2019-11-16 17:23:00,NaN,NaN


In [14]:
# ------------------------------------------------------------
# 4) Put ice-edge distance into xarray and align to ds_out time
# ------------------------------------------------------------
ice_dist_xr = xr.Dataset(
    data_vars=dict(
        ice_edge_dist_m=("time", ice_dist_df["ice_edge_dist_m"].values),
        ice_edge_dist_km=("time", ice_dist_df["ice_edge_dist_km"].values),
    ),
    coords=dict(
        time=pd.to_datetime(ice_dist_df["time"].values)
    )
)

# align sparse ice-edge times to the mooring 30-min time grid
ice_dist_on_moor_time = ice_dist_xr.reindex(
    time=ds_out.time,
    method="nearest",
    tolerance=pd.Timedelta("1h")
)

ice_dist_on_moor_time

<xarray.Dataset> Size: 17kB
Dimensions:           (time: 718)
Coordinates:
  * time              (time) datetime64[ns] 6kB 2019-11-09T20:30:00 ... 2019-...
Data variables:
    ice_edge_dist_m   (time) float64 6kB nan nan nan nan nan ... nan nan nan nan
    ice_edge_dist_km  (time) float64 6kB nan nan nan nan nan ... nan nan nan nan

In [15]:
source_order = list(ds_out.source.values)
moor_meta = moor_line_gdf.set_index("source").loc[source_order]

# Extend time to cover both mooring data and ice edge data
full_time = pd.date_range(
    start = ds_out.time.values.min(),
    end   = max(ds_out.time.values.max(), ice_dist_xr.time.values.max()),
    freq  = "30min"
)

final_ds = xr.Dataset(
    coords=dict(
        source=ds_out.source,
        time=full_time
    )
)

# wave height — reindex to full time grid so NaN fills the extended period
final_ds["wave_height_m"] = ds_out["sigwaveheight"].reindex(time=full_time)

# fixed mooring distances (by source)
final_ds["moor_dist_m"] = xr.DataArray(
    moor_meta["moor_dist_m"].values,
    dims=("source",),
    coords={"source": source_order}
)
final_ds["moor_dist_km"] = xr.DataArray(
    moor_meta["moor_dist_km"].values,
    dims=("source",),
    coords={"source": source_order}
)

# optional lat/lon metadata
final_ds["moor_lat"] = xr.DataArray(
    moor_meta["lat"].values,
    dims=("source",),
    coords={"source": source_order}
)
final_ds["moor_lon"] = xr.DataArray(
    moor_meta["lon"].values,
    dims=("source",),
    coords={"source": source_order}
)

# time-varying ice-edge distance — reindex onto full time grid
ice_dist_on_moor_time = ice_dist_xr.reindex(
    time      = full_time,
    method    = "nearest",
    tolerance = pd.Timedelta("1h")
)

final_ds["ice_edge_dist_m"]  = ice_dist_on_moor_time["ice_edge_dist_m"]
final_ds["ice_edge_dist_km"] = ice_dist_on_moor_time["ice_edge_dist_km"]

final_ds.attrs["crs"]            = "EPSG:32604"
final_ds.attrs["reference_axis"] = f"Distance along mooring transect, zeroed at {zero_point}"

print("final_ds time range:", final_ds.time.min().values, "to", final_ds.time.max().values)
final_ds

final_ds time range: 2019-11-09T20:30:00.000000000 to 2019-11-29T04:00:00.000000000


<xarray.Dataset> Size: 52kB
Dimensions:           (source: 4, time: 928)
Coordinates:
  * source            (source) <U4 64B 'S1P1' 'S1P2' 'S1P3' 'S1A1'
  * time              (time) datetime64[ns] 7kB 2019-11-09T20:30:00 ... 2019-...
Data variables:
    wave_height_m     (source, time) float64 30kB nan nan nan ... nan nan nan
    moor_dist_m       (source) float64 32B 5.929e+03 1.212e+04 ... 2.377e+04
    moor_dist_km      (source) float64 32B 5.929 12.12 17.74 23.77
    moor_lat          (source) float64 32B 70.35 70.39 70.44 70.49
    moor_lon          (source) float64 32B -162.1 -162.1 -162.2 -162.3
    ice_edge_dist_m   (time) float64 7kB nan nan nan nan nan ... nan nan nan nan
    ice_edge_dist_km  (time) float64 7kB nan nan nan nan nan ... nan nan nan nan
Attributes:
    crs:             EPSG:32604
    reference_axis:  Distance along mooring transect, zeroed at POINT (387833...

In [16]:
final_ds["ice_edge_dist_m"] = final_ds["ice_edge_dist_m"].interpolate_na(dim="time", method="linear")

# For each source, calculate dist_from_ice = ice_edge_dist_m - moor_dist_m at every time step
dist_from_ice = (
    final_ds["ice_edge_dist_m"] - final_ds["moor_dist_m"]
)  # broadcasts automatically: (time,) - (source,) -> (source, time)

final_ds["dist_from_ice"] = dist_from_ice
final_ds["dist_from_ice"].attrs = {
    "long_name": "Distance from mooring to ice edge",
    "units": "m"
}

final_ds

<xarray.Dataset> Size: 82kB
Dimensions:           (source: 4, time: 928)
Coordinates:
  * source            (source) <U4 64B 'S1P1' 'S1P2' 'S1P3' 'S1A1'
  * time              (time) datetime64[ns] 7kB 2019-11-09T20:30:00 ... 2019-...
Data variables:
    wave_height_m     (source, time) float64 30kB nan nan nan ... nan nan nan
    moor_dist_m       (source) float64 32B 5.929e+03 1.212e+04 ... 2.377e+04
    moor_dist_km      (source) float64 32B 5.929 12.12 17.74 23.77
    moor_lat          (source) float64 32B 70.35 70.39 70.44 70.49
    moor_lon          (source) float64 32B -162.1 -162.1 -162.2 -162.3
    ice_edge_dist_m   (time) float64 7kB nan nan nan nan nan ... nan nan nan nan
    ice_edge_dist_km  (time) float64 7kB nan nan nan nan nan ... nan nan nan nan
    dist_from_ice     (time, source) float64 30kB nan nan nan ... nan nan nan
Attributes:
    crs:             EPSG:32604
    reference_axis:  Distance along mooring transect, zeroed at POINT (387833...

In [17]:
final_df = final_ds.to_dataframe().reset_index()
final_df

,source,time,wave_height_m,moor_dist_m,moor_dist_km,moor_lat,moor_lon,ice_edge_dist_m,ice_edge_dist_km,dist_from_ice
0,S1P1,2019-11-09 20:30:00,NaN,5928.534141,5.928534,70.34613,-162.05704,NaN,NaN,NaN
1,S1P1,2019-11-09 21:00:00,NaN,5928.534141,5.928534,70.34613,-162.05704,NaN,NaN,NaN
2,S1P1,2019-11-09 21:30:00,NaN,5928.534141,5.928534,70.34613,-162.05704,NaN,NaN,NaN
3,S1P1,2019-11-09 22:00:00,NaN,5928.534141,5.928534,70.34613,-162.05704,NaN,NaN,NaN
4,S1P1,2019-11-09 22:30:00,NaN,5928.534141,5.928534,70.34613,-162.05704,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
3707,S1A1,2019-11-29 02:00:00,NaN,23768.264612,23.768265,70.48695,-162.28278,NaN,NaN,NaN
3708,S1A1,2019-11-29 02:30:00,NaN,23768.264612,23.768265,70.48695,-162.28278,NaN,NaN,NaN
3709,S1A1,2019-11-29 03:00:00,NaN,23768.264612,23.768265,70.48695,-162.28278,NaN,NaN,NaN
3710,S1A1,2019-11-29 03:30:00,NaN,23768.264612,23.768265,70.48695,-162.28278,NaN,NaN,NaN


In [18]:
#Save and Export
final_df.to_csv("/home/jovyan/Society_of_Bouy_Cowboys/Data/Organized_Data/final_df.csv", index=False)
final_ds.to_netcdf("/home/jovyan/Society_of_Bouy_Cowboys/Data/Organized_Data/final_ds.nc")